In [1]:
import os
import pandas as pd
import numpy as np
import re
from rapidfuzz import process, fuzz

from use.config import DATA_PATH, COMPS, DES_SEASONS
from use.functions import REPLACEMENTS, json_to_dict, create_slug, need_to_upload, elapsed_time_str, generate_unique_ids

# Estructura de carpetas
RAW_DATA_PATH = os.path.join(DATA_PATH, "raw")
BRONZE_DATA_PATH = os.path.join(DATA_PATH, "bronze")
os.makedirs(BRONZE_DATA_PATH, exist_ok=True)

import ss_cln as ss
import sw_cln as sw

tot aixo a unify

In [2]:
# --------------------------------------------------------------------------------------
# NORMALIZACIÓN
# --------------------------------------------------------------------------------------
def normalize_name(text: str) -> str:
    if not isinstance(text, str):
        return ""

    for k, v in REPLACEMENTS.items():
        text = text.replace(k, v)

    text = text.lower()
    text = re.sub(r'\b(fc|cf|club|de|la|ud|cd|sd|fk|ac|sc)\b', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# --------------------------------------------------------------------------------------
# LIMPIEZA DEL DATAFRAME DE VENUES
# --------------------------------------------------------------------------------------
def clean_venues_df(df: pd.DataFrame) -> pd.DataFrame:

    df_cleaned = df.copy()
    df_cleaned = df_cleaned[["Slug", "Name", "Capacity", "City", "Latitude", "Longitude", "IdSS"]]
    df_cleaned.insert(0, "ID", generate_unique_ids(len(df_cleaned)))
    return df_cleaned.sort_values(by="Slug").reset_index(drop=True)

# --------------------------------------------------------------------------------------
# MATCHING EQUIPOS DE UNA LIGA
# --------------------------------------------------------------------------------------
def matching_part_teams(ss_teams_part: pd.DataFrame, sw_teams_part: pd.DataFrame, threshold: int = 75) -> pd.DataFrame:

    ss_teams_part = ss_teams_part.copy()
    sw_teams_part = sw_teams_part.copy()

    # Normalizamos para matching
    for col in ["Slug", "Name", "ShortName", "LongName"]:
        ss_teams_part[col] = ss_teams_part[col].fillna("").apply(normalize_name)
        sw_teams_part[col] = sw_teams_part[col].fillna("").apply(normalize_name)

    # Siempre A -> B desde el mayor al menor
    if len(ss_teams_part) > len(sw_teams_part):
        df_A = ss_teams_part
        df_B = sw_teams_part
        cols_dict = {"SlugA":"SlugSS", "SlugB":"SlugSW",
                     "NameA":"NameSS", "NameB":"NameSW",
                     "ShortNameA":"ShortNameSS", "ShortNameB":"ShortNameSW",
                     "LongNameA":"LongNameSS", "LongNameB":"LongNameSW"}
    else:
        df_A = sw_teams_part
        df_B = ss_teams_part
        cols_dict = {"SlugA":"SlugSW", "SlugB":"SlugSS",
                     "NameA":"NameSW", "NameB":"NameSS",
                     "ShortNameA":"ShortNameSW", "ShortNameB":"ShortNameSS",
                     "LongNameA":"LongNameSW", "LongNameB":"LongNameSS"}

    choices_slug = [x for x in df_B["Slug"].dropna().unique().tolist() if x != ""]
    choices_name = [x for x in df_B["Name"].dropna().unique().tolist() if x != ""]
    choices_s_name = [x for x in df_B["ShortName"].dropna().unique().tolist() if x != ""]
    choices_l_name = [x for x in df_B["LongName"].dropna().unique().tolist() if x != ""]

    results = []

    for _, row in df_A.iterrows():

        row_slug = row["Slug"]
        row_name = row["Name"]
        row_s_name = row["ShortName"]
        row_l_name = row["LongName"]

        match_slug, score_slug = (np.nan, 0)
        match_name, score_name = (np.nan, 0)
        match_s_name, score_s_name = (np.nan, 0)
        match_l_name, score_l_name = (np.nan, 0)

        if row_slug and choices_slug:
            tmp = process.extractOne(row_slug, choices_slug, scorer=fuzz.token_set_ratio)
            if tmp:
                match_slug, score_slug, _ = tmp

        if row_name and choices_name:
            tmp = process.extractOne(row_name, choices_name, scorer=fuzz.token_set_ratio)
            if tmp:
                match_name, score_name, _ = tmp

        if row_s_name and choices_s_name:
            tmp = process.extractOne(row_s_name, choices_s_name, scorer=fuzz.token_set_ratio)
            if tmp:
                match_s_name, score_s_name, _ = tmp

        if row_l_name and choices_l_name:
            tmp = process.extractOne(row_l_name, choices_l_name, scorer=fuzz.token_set_ratio)
            if tmp:
                match_l_name, score_l_name, _ = tmp

        results.append({"SlugA": row_slug, "SlugB": match_slug if score_slug >= threshold else np.nan, "ScoreSlug": score_slug,
                        "NameA": row_name, "NameB": match_name if score_name >= threshold else np.nan, "ScoreName": score_name,
                        "ShortNameA": row_s_name, "ShortNameB": match_s_name if score_s_name >= threshold else np.nan, "ScoreShortName": score_s_name,
                        "LongNameA": row_l_name, "LongNameB": match_l_name if score_l_name >= threshold else np.nan, "ScoreLongName": score_l_name})

    matched_teams = pd.DataFrame(results).rename(columns=cols_dict)

    # Diccionarios de lookup SOLO con los part de la liga
    ss_slug_to_id = ss_teams_part.drop_duplicates("Slug").set_index("Slug")["IdSS"].to_dict()
    ss_name_to_id = ss_teams_part.drop_duplicates("Name").set_index("Name")["IdSS"].to_dict()
    ss_sname_to_id = ss_teams_part.drop_duplicates("ShortName").set_index("ShortName")["IdSS"].to_dict()
    ss_lname_to_id = ss_teams_part.drop_duplicates("LongName").set_index("LongName")["IdSS"].to_dict()

    sw_slug_to_id = sw_teams_part.drop_duplicates("Slug").set_index("Slug")["IdSW"].to_dict()
    sw_name_to_id = sw_teams_part.drop_duplicates("Name").set_index("Name")["IdSW"].to_dict()
    sw_sname_to_id = sw_teams_part.drop_duplicates("ShortName").set_index("ShortName")["IdSW"].to_dict()
    sw_lname_to_id = sw_teams_part.drop_duplicates("LongName").set_index("LongName")["IdSW"].to_dict()

    ids_results = []

    for _, row in matched_teams.iterrows():

        candidates = [("Slug", row["ScoreSlug"], row["SlugSS"], row["SlugSW"]), ("Name", row["ScoreName"], row["NameSS"], row["NameSW"]),
                      ("ShortName", row["ScoreShortName"], row["ShortNameSS"], row["ShortNameSW"]), ("LongName", row["ScoreLongName"], row["LongNameSS"], row["LongNameSW"])]

        # solo candidatos con ambos valores no nulos
        candidates = [c for c in candidates if pd.notna(c[2]) and pd.notna(c[3])]

        if len(candidates) == 0:
            ids_results.append({"IdSS": np.nan, "IdSW": np.nan})
            continue

        best_field, _, ss_value, sw_value = max(candidates, key=lambda x: x[1])

        if best_field == "Slug":
            id_ss = ss_slug_to_id.get(ss_value, np.nan)
            id_sw = sw_slug_to_id.get(sw_value, np.nan)
        elif best_field == "Name":
            id_ss = ss_name_to_id.get(ss_value, np.nan)
            id_sw = sw_name_to_id.get(sw_value, np.nan)
        elif best_field == "ShortName":
            id_ss = ss_sname_to_id.get(ss_value, np.nan)
            id_sw = sw_sname_to_id.get(sw_value, np.nan)
        else:
            id_ss = ss_lname_to_id.get(ss_value, np.nan)
            id_sw = sw_lname_to_id.get(sw_value, np.nan)

        ids_results.append({"IdSS": id_ss, "IdSW": id_sw})

    matched_teams_ids = pd.DataFrame(ids_results).drop_duplicates()

    # IDs no mapeados
    not_matched_ss_ids = [i for i in ss_teams_part["IdSS"] if i not in matched_teams_ids["IdSS"].dropna().tolist()]
    not_matched_sw_ids = [i for i in sw_teams_part["IdSW"] if i not in matched_teams_ids["IdSW"].dropna().tolist()]
    df_not_matched_ss = pd.DataFrame({"IdSS": not_matched_ss_ids, "IdSW": np.nan})
    df_not_matched_sw = pd.DataFrame({"IdSS": np.nan, "IdSW": not_matched_sw_ids})

    matched_teams_ids = pd.concat([matched_teams_ids, df_not_matched_ss, df_not_matched_sw], ignore_index=True).drop_duplicates()

    return matched_teams_ids

# --------------------------------------------------------------------------------------
# LIMPIEZA DEL DATAFRAME DE EQUIPOS MAPEADO
# --------------------------------------------------------------------------------------
def clean_matched_teams_df(df: pd.DataFrame) -> pd.DataFrame:

    # Dataframe y columnas
    df_cleaned = pd.DataFrame(index=df.index)
    cols_map = {"Name": ["NameSS", "NameSW"], "ShortName": ["ShortNameSS", "ShortNameSW"], "LongName": ["LongNameSS", "LongNameSW"], "Abbreviation": ["AbbreviationSW"], "Country": ["CountrySS"], 
                "FoundationDate": ["FoundationDateSS"], "Manager": ["ManagerSS"], "Venue": ["VenueSS"], "PrimaryColour": ["PrimaryColourSS"], "SecondaryColour": ["SecondaryColourSS"],
                "TextColour": ["TextColourSS"], "HomeKitCol1": ["HomeKitCol1SW"], "HomeKitCol2": ["HomeKitCol2SW"], "HomeShortsCol": ["HomeShortsColSW"], "AwayKitCol1": ["AwayKitCol1SW"], 
                "AwayKitCol2": ["AwayKitCol2SW"], "AwayShortsCol": ["AwayShortsColSW"], "IdSS": ["IdSS"], "IdSW": ["IdSW"]}

    # Para cada columna, la añadimos
    for new_col, possible_cols in cols_map.items():
        existing_cols = [col for col in possible_cols if col in df.columns]
        if existing_cols:
            df_cleaned[new_col] = df[existing_cols].bfill(axis=1).iloc[:, 0]
        else:
            df_cleaned[new_col] = np.nan

    df_cleaned.insert(0, "Slug", df_cleaned["Name"].apply(create_slug))
    df_cleaned.insert(0, "ID", generate_unique_ids(len(df_cleaned)))

    return df_cleaned.sort_values(by="Slug").reset_index(drop=True)

# --------------------------------------------------------------------------------------
# MATCHING EQUIPOS - Matching de todos los equipos a partir de los dataframes de equipos y partidos
# --------------------------------------------------------------------------------------
def matched_teams_df(ss_matches: pd.DataFrame, sw_matches: pd.DataFrame, ss_teams: pd.DataFrame, sw_teams: pd.DataFrame) -> pd.DataFrame:

    all_teams_matched = []

    for league in COMPS["tournament"].tolist():

        league_slug = create_slug(text=league)

        # Equipos SS
        ss_matches_part = ss_matches[ss_matches["League"] == league_slug]
        ss_teams_list = list(set(ss_matches_part["HomeTeam"].dropna().tolist() + ss_matches_part["AwayTeam"].dropna().tolist()))
        ss_teams_part = ss_teams[ss_teams["IdSS"].isin(ss_teams_list)][["IdSS", "Slug", "Name", "ShortName", "LongName"]].drop_duplicates()

        # Equipos SW
        sw_matches_part = sw_matches[sw_matches["League"] == league_slug]
        sw_teams_list = list(set(sw_matches_part["HomeTeam"].dropna().tolist() + sw_matches_part["AwayTeam"].dropna().tolist()))
        sw_teams_part = sw_teams[sw_teams["IdSW"].isin(sw_teams_list)][["IdSW", "Slug", "Name", "ShortName", "LongName"]].drop_duplicates()

        # Obtenemos mapeo y añadimos a la lista
        if not ss_teams_part.empty and not sw_teams_part.empty:
            matched_codes_df = matching_part_teams(ss_teams_part=ss_teams_part, sw_teams_part=sw_teams_part)
            all_teams_matched.append(matched_codes_df)

    all_teams_matched_df = pd.concat(all_teams_matched, ignore_index=True).drop_duplicates()

    # Resolver duplicados por IdSS
    all_teams_matched_df = (all_teams_matched_df.sort_values(by=["IdSS", "IdSW"], na_position="last").groupby("IdSS", dropna=False)["IdSW"]
                            .apply(lambda x: x.dropna().iloc[0] if x.dropna().any() else np.nan).reset_index())

    # Resolver duplicados por IdSW
    all_teams_matched_df = (all_teams_matched_df.sort_values(by=["IdSW", "IdSS"], na_position="last").groupby("IdSW", dropna=False)["IdSS"]
                            .apply(lambda x: x.dropna().iloc[0] if x.dropna().any() else np.nan).reset_index())

    # Para cada ID concatenamos la información
    list_teams_dfs = []
    for _, row in all_teams_matched_df.iterrows():

        id_ss = row["IdSS"]
        id_sw = row["IdSW"]

        team_ss = ss_teams[ss_teams["IdSS"] == id_ss].copy()
        team_sw = sw_teams[sw_teams["IdSW"] == id_sw].copy()

        if team_ss.empty:
            team_ss = pd.DataFrame([{"IdSS": id_ss}])
        else:
            team_ss = team_ss.rename(columns=lambda col: col if col == "IdSS" else f"{col}SS")

        if team_sw.empty:
            team_sw = pd.DataFrame([{"IdSW": id_sw}])
        else:
            team_sw = team_sw.rename(columns=lambda col: col if col == "IdSW" else f"{col}SW")

        team_ss = team_ss.reset_index(drop=True)
        team_sw = team_sw.reset_index(drop=True)

        team_df = pd.concat([team_ss, team_sw], axis=1)
        list_teams_dfs.append(team_df)

    # Concatenamos todos los equipos
    all_teams_df = pd.concat(list_teams_dfs, ignore_index=True)

    # Dataframes con equipos no mapeados
    not_matched_ss = ss_teams[~ss_teams["IdSS"].isin(all_teams_matched_df["IdSS"].unique().tolist())]
    not_matched_sw = sw_teams[~sw_teams["IdSW"].isin(all_teams_matched_df["IdSW"].unique().tolist())]

    # Añadir SS y SW a las columnas
    not_matched_ss = not_matched_ss.rename(columns=lambda col: col if col == "IdSS" else f"{col}SS")
    not_matched_sw = not_matched_sw.rename(columns=lambda col: col if col == "IdSW" else f"{col}SW")

    # Concatenamos con all teams
    all_teams_df = pd.concat([all_teams_df, not_matched_ss, not_matched_sw], ignore_index=True)

    return clean_matched_teams_df(all_teams_df)     # Return del dataframe limpio

# --------------------------------------------------------------------------------------
# MATCHING DE JUGADORES DE UN EQUIPO
# --------------------------------------------------------------------------------------
def matching_part_players(ss_players_part: pd.DataFrame, sw_players_part: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:

    ss_players_ = ss_players_part[["IdSS", "Slug", "Name", "ShortName"]].drop_duplicates().copy()
    sw_players_ = sw_players_part[["IdSW", "Slug", "Name", "MatchName"]].rename(columns={"MatchName": "ShortName"}).drop_duplicates().copy()

    # Normalizamos para matching
    for col in ["Name", "ShortName"]:
        ss_players_[col] = ss_players_[col].fillna("").apply(normalize_name)
        sw_players_[col] = sw_players_[col].fillna("").apply(normalize_name)

    # Siempre A -> B desde el mayor al menor
    if len(ss_players_) > len(sw_players_):
        df_A = ss_players_
        df_B = sw_players_
        cols_dict = {"SlugA":"SlugSS", "SlugB":"SlugSW",
                    "NameA":"NameSS", "NameB":"NameSW",
                    "ShortNameA":"ShortNameSS", "ShortNameB":"ShortNameSW"}
    else:
        df_A = sw_players_
        df_B = ss_players_
        cols_dict = {"SlugA":"SlugSW", "SlugB":"SlugSS",
                    "NameA":"NameSW", "NameB":"NameSS",
                    "ShortNameA":"ShortNameSW", "ShortNameB":"ShortNameSS"}
        
    choices_slug = [x for x in df_B["Slug"].dropna().unique().tolist() if x != ""]
    choices_name = [x for x in df_B["Name"].dropna().unique().tolist() if x != ""]
    choices_s_name = [x for x in df_B["ShortName"].dropna().unique().tolist() if x != ""]

    results = []

    # Para cada jugador del dataframe mas grande
    for _, row in df_A.iterrows():

        row_slug = row["Slug"]
        row_name = row["Name"]
        row_s_name = row["ShortName"]

        match_slug, score_slug = (np.nan, 0)
        match_name, score_name = (np.nan, 0)
        match_s_name, score_s_name = (np.nan, 0)

        if row_slug and choices_slug:
            tmp = process.extractOne(row_slug, choices_slug, scorer=fuzz.token_set_ratio)
            if tmp:
                match_slug, score_slug, _ = tmp

        if row_name and choices_name:
            tmp = process.extractOne(row_name, choices_name, scorer=fuzz.token_set_ratio)
            if tmp:
                match_name, score_name, _ = tmp

        if row_s_name and choices_s_name:
            tmp = process.extractOne(row_s_name, choices_s_name, scorer=fuzz.token_set_ratio)
            if tmp:
                match_s_name, score_s_name, _ = tmp

        results.append({"SlugA": row_slug, "SlugB": match_slug if score_slug >= threshold else np.nan, "ScoreSlug": score_slug,
                        "NameA": row_name, "NameB": match_name if score_name >= threshold else np.nan, "ScoreName": score_name,
                        "ShortNameA": row_s_name, "ShortNameB": match_s_name if score_s_name >= threshold else np.nan, "ScoreShortName": score_s_name})
        
    # Jugadores mapeados
    matched_players = pd.DataFrame(results).rename(columns=cols_dict)

    # Diccionarios de lookup SOLO con los part de la liga
    ss_slug_to_id = ss_players_.drop_duplicates("Slug").set_index("Slug")["IdSS"].to_dict()
    ss_name_to_id = ss_players_.drop_duplicates("Name").set_index("Name")["IdSS"].to_dict()
    ss_sname_to_id = ss_players_.drop_duplicates("ShortName").set_index("ShortName")["IdSS"].to_dict()

    sw_slug_to_id = sw_players_.drop_duplicates("Slug").set_index("Slug")["IdSW"].to_dict()
    sw_name_to_id = sw_players_.drop_duplicates("Name").set_index("Name")["IdSW"].to_dict()
    sw_sname_to_id = sw_players_.drop_duplicates("ShortName").set_index("ShortName")["IdSW"].to_dict()

    ids_results = []

    for _, row in matched_players.iterrows():

        candidates = [("Slug", row["ScoreSlug"], row["SlugSS"], row["SlugSW"]), 
                    ("Name", row["ScoreName"], row["NameSS"], row["NameSW"]),
                    ("ShortName", row["ScoreShortName"], row["ShortNameSS"], row["ShortNameSW"])]

        # solo candidatos con ambos valores no nulos
        candidates = [c for c in candidates if pd.notna(c[2]) and pd.notna(c[3])]

        if len(candidates) == 0:
            ids_results.append({"IdSS": np.nan, "IdSW": np.nan})
            continue

        best_field, _, ss_value, sw_value = max(candidates, key=lambda x: x[1])

        if best_field == "Slug":
            id_ss = ss_slug_to_id.get(ss_value, np.nan)
            id_sw = sw_slug_to_id.get(sw_value, np.nan)
        elif best_field == "Name":
            id_ss = ss_name_to_id.get(ss_value, np.nan)
            id_sw = sw_name_to_id.get(sw_value, np.nan)
        elif best_field == "ShortName":
            id_ss = ss_sname_to_id.get(ss_value, np.nan)
            id_sw = sw_sname_to_id.get(sw_value, np.nan)

        ids_results.append({"IdSS": id_ss, "IdSW": id_sw})

    matched_players_ids = pd.DataFrame(ids_results).drop_duplicates().dropna(subset=["IdSS", "IdSW"])
    return matched_players_ids

# --------------------------------------------------------------------------------------
# LIMPIEZA DEL DATAFRAME DE JUGADORES MAPEADO
# --------------------------------------------------------------------------------------
def clean_matched_players_df(df: pd.DataFrame) -> pd.DataFrame:

    # Dataframe y columnas
    df_cleaned = pd.DataFrame(index=df.index)
    cols_map = {"Name": ["NameSS", "NameSW"], "ShortName": ["ShortNameSW"], "FirstName": ["FirstNameSW"], "LastName": ["LastNameSW"], "ShortFirstName": ["ShortFirstNameSW"], 
                "ShortLastName": ["ShortLastNameSW"], "MatchName": ["ShortNameSS", "MatchNameSW"], "Country": ["CountrySS", "CountrySW"], "DateBirth": ["DateBirthSS"], 
                "Height": ["HeightSS"], "PrefFoot": ["PrefFootSS"], "Position": ["PositionSW"], "FirstPos": ["FirstPositionSS"], "SecondPos": ["SecondPositionSS"], 
                "ThirdPos": ["ThirdPositionSS"], "ContractUntil": ["ContractUntilSS"], "MarketValue": ["MarketValueSS"], "IdSS": ["IdSS"], "IdSW": ["IdSW"]}

    # Para cada columna, la añadimos
    for new_col, possible_cols in cols_map.items():
        existing_cols = [col for col in possible_cols if col in df.columns]
        if existing_cols:
            df_cleaned[new_col] = df[existing_cols].bfill(axis=1).iloc[:, 0]
        else:
            df_cleaned[new_col] = np.nan

    df_cleaned.insert(0, "Slug", df_cleaned["Name"].apply(create_slug))
    df_cleaned.insert(3, "LongName", df_cleaned["FirstName"] + " " + df_cleaned["LastName"])
    df_cleaned.insert(0, "ID", generate_unique_ids(len(df_cleaned)))
    
    return df_cleaned.sort_values(by="Slug").reset_index(drop=True)

# --------------------------------------------------------------------------------------
# MATCHING DE JUGADORES - Matching de todos los jugadores, concatenamos por equipo
# --------------------------------------------------------------------------------------
def matched_players_df(ss_players: pd.DataFrame, sw_players: pd.DataFrame, teams_df: pd.DataFrame) -> pd.DataFrame:
    
    all_players_matched = []

    for _, row in teams_df.iterrows():

        # Obtenemos los jugadores mapeados por equipo
        id_ss = row["IdSS"]
        id_sw = row["IdSW"]
        team_ss = ss_players[ss_players["Team"] == id_ss].copy()
        team_sw = sw_players[sw_players["Team"] == id_sw].copy()

        if team_ss.empty or team_sw.empty:
            continue
        else:
            matched_players = matching_part_players(ss_players_part=team_ss, sw_players_part=team_sw)
            all_players_matched.append(matched_players) 

    # Dataframe
    all_players_matched_df = pd.concat(all_players_matched, ignore_index=True)

    list_players_dfs = []
    for _, row in all_players_matched_df.iterrows():

        id_ss = row["IdSS"]
        id_sw = row["IdSW"]

        player_ss = ss_players[ss_players["IdSS"] == id_ss].copy()
        player_sw = sw_players[sw_players["IdSW"] == id_sw].drop_duplicates(subset="IdSW").copy()

        if player_ss.empty:
            player_ss = pd.DataFrame([{"IdSS": id_ss}])
        else:
            player_ss = player_ss.rename(columns=lambda col: col if col == "IdSS" else f"{col}SS")

        if player_sw.empty:
            player_sw = pd.DataFrame([{"IdSW": id_sw}])
        else:
            player_sw = player_sw.rename(columns=lambda col: col if col == "IdSW" else f"{col}SW")

        player_ss = player_ss.reset_index(drop=True)
        player_sw = player_sw.reset_index(drop=True)

        player_df = pd.concat([player_ss, player_sw], axis=1)
        list_players_dfs.append(player_df)

    # Dataframe
    players_df = pd.concat(list_players_dfs, ignore_index=True)

    # Dataframes con jugadores no mapeados
    not_matched_ss = ss_players[~ss_players["IdSS"].isin(all_players_matched_df["IdSS"].unique().tolist())]
    not_matched_sw = sw_players[~sw_players["IdSW"].isin(all_players_matched_df["IdSW"].unique().tolist())]

    # Añadir SS y SW a las columnas
    not_matched_ss = not_matched_ss.rename(columns=lambda col: col if col == "IdSS" else f"{col}SS").drop_duplicates(subset="IdSS")
    not_matched_sw = not_matched_sw.rename(columns=lambda col: col if col == "IdSW" else f"{col}SW").drop_duplicates(subset="IdSW")

    # Concatenación
    all_players_df = pd.concat([players_df, not_matched_ss, not_matched_sw], ignore_index=True)

    return clean_matched_players_df(all_players_df)

# --------------------------------------------------------------------------------------
# LIMPIEZA DEL DATAFRAME DE ENTRENADORES MAPEADO
# --------------------------------------------------------------------------------------
def clean_matched_managers_df(df: pd.DataFrame) -> pd.DataFrame:

    # Dataframe y columnas
    df_cleaned = pd.DataFrame(index=df.index)
    cols_map = {"Name": ["NameSS", "NameSW"], "ShortName": ["ShortNameSW"], "FirstName": ["FirstNameSW"], "LastName": ["LastNameSW"], "ShortFirstName": ["ShortFirstNameSW"], 
                "ShortLastName": ["ShortLastNameSW"], "MatchName": ["ShortNameSS", "MatchNameSW"], "Country": ["CountrySS", "CountrySW"], "DateBirth": ["DateBirthSS"], 
                "Type": ["TypeSW"], "Matches": ["MatchesSS"], "Wins": ["WinsSS"], "Draws": ["DrawsSS"], "Losses": ["LossesSS"], "GoalsFor": ["GoalsForSS"], 
                "GoalsAgainst": ["GoalsAgainstSS"], "Points": ["PointsSS"], "IdSS": ["IdSS"], "IdSW": ["IdSW"]}

    # Para cada columna, la añadimos
    for new_col, possible_cols in cols_map.items():
        existing_cols = [col for col in possible_cols if col in df.columns]
        if existing_cols:
            df_cleaned[new_col] = df[existing_cols].bfill(axis=1).iloc[:, 0]
        else:
            df_cleaned[new_col] = np.nan

    df_cleaned.insert(0, "Slug", df_cleaned["Name"].apply(create_slug))
    df_cleaned.insert(3, "LongName", df_cleaned["FirstName"] + " " + df_cleaned["LastName"])
    df_cleaned.insert(0, "ID", generate_unique_ids(len(df_cleaned)))

    return df_cleaned.sort_values(by="Slug").reset_index(drop=True)

# --------------------------------------------------------------------------------------
# MATCHING DE ENTRENADORES - Matching de todos los entrenadores
# --------------------------------------------------------------------------------------
def matched_managers_df(ss_managers: pd.DataFrame, sw_managers: pd.DataFrame) -> pd.DataFrame:

    # Obtenemos los entrenadores mapeados a partir de la función de jugadores - también se aplica
    matched_managers_ids = matching_part_players(ss_players_part=ss_managers, sw_players_part=sw_managers)

    list_managers_dfs = []
    for _, row in matched_managers_ids.iterrows():

        id_ss = row["IdSS"]
        id_sw = row["IdSW"]

        manager_ss = ss_managers[ss_managers["IdSS"] == id_ss].copy()
        manager_sw = sw_managers[sw_managers["IdSW"] == id_sw].drop_duplicates(subset="IdSW").copy()

        if manager_ss.empty:
            manager_ss = pd.DataFrame([{"IdSS": id_ss}])
        else:
            manager_ss = manager_ss.rename(columns=lambda col: col if col == "IdSS" else f"{col}SS")

        if manager_sw.empty:
            manager_sw = pd.DataFrame([{"IdSW": id_sw}])
        else:
            manager_sw = manager_sw.rename(columns=lambda col: col if col == "IdSW" else f"{col}SW")

        manager_ss = manager_ss.reset_index(drop=True)
        manager_sw = manager_sw.reset_index(drop=True)

        manager_df = pd.concat([manager_ss, manager_sw], axis=1)
        list_managers_dfs.append(manager_df)

    # Dataframe
    managers_df = pd.concat(list_managers_dfs)

    # Dataframes con jugadores no mapeados
    not_matched_ss = ss_managers[~ss_managers["IdSS"].isin(matched_managers_ids["IdSS"].unique().tolist())]
    not_matched_sw = sw_managers[~sw_managers["IdSW"].isin(matched_managers_ids["IdSW"].unique().tolist())]

    # Añadir SS y SW a las columnas
    not_matched_ss = not_matched_ss.rename(columns=lambda col: col if col == "IdSS" else f"{col}SS").drop_duplicates(subset="IdSS")
    not_matched_sw = not_matched_sw.rename(columns=lambda col: col if col == "IdSW" else f"{col}SW").drop_duplicates(subset="IdSW")

    # Concatenación
    all_managers_df = pd.concat([managers_df, not_matched_ss, not_matched_sw], ignore_index=True)

    return clean_matched_managers_df(all_managers_df)

# --------------------------------------------------------------------------------------
# LIMPIEZA DEL DATAFRAME DE PARTIDOS MAPEADOS
# --------------------------------------------------------------------------------------
def clean_matched_matches_df(df: pd.DataFrame) -> pd.DataFrame:

    # Dataframe y columnas
    df_cleaned = pd.DataFrame(index=df.index)
    cols_map = {"League": ["League"], "Season": ["Season"], "Round": ["RoundSS"], "HomeTeam": ["HomeTeamID"], "AwayTeam": ["AwayTeamID"], "Date": ["DateSS", "DateSW"], 
                "Time": ["TimeSS", "TimeSW"], "Venue": ["VenueSS"], "Referee": ["RefereeSS"], "HomeScore": ["HomeScoreSS"], "AwayScore": ["AwayScoreSS"],
                "IdSS": ["IdSS"], "IdSW": ["IdSW"]}

    # Para cada columna, la añadimos
    for new_col, possible_cols in cols_map.items():
        existing_cols = [col for col in possible_cols if col in df.columns]
        if existing_cols:
            df_cleaned[new_col] = df[existing_cols].bfill(axis=1).iloc[:, 0]
        else:
            df_cleaned[new_col] = np.nan

    df_cleaned.insert(0, "ID", generate_unique_ids(len(df_cleaned)))

    return df_cleaned.sort_values(by=["League", "Season", "Round", "Date", "Time"]).reset_index(drop=True)

# --------------------------------------------------------------------------------------
# MATCHING DE PARTIDOS - A partir de equipos y partidos
# --------------------------------------------------------------------------------------
def matched_matches_df(ss_matches: pd.DataFrame, sw_matches: pd.DataFrame, teams_df: pd.DataFrame) -> pd.DataFrame:

    # Obtenemos diccionario con el Id de la fuente y el Id del equipo interno
    ss_to_id = teams_df.dropna(subset=["IdSS"]).drop_duplicates(subset="IdSS").set_index("IdSS")["ID"].to_dict()
    sw_to_id = teams_df.dropna(subset=["IdSW"]).drop_duplicates(subset="IdSW").set_index("IdSW")["ID"].to_dict()

    all_matches_list = []

    # Para cada liga y temporada
    for league in COMPS["tournament"].tolist():
        for season in DES_SEASONS:

            league_slug = create_slug(text=league)
            
            # Mapeamos equipo local y equipo visitante - SW
            sw_matches_ = sw_matches[(sw_matches["League"] == league_slug) & (sw_matches["Season"] == str(season))].copy()
            sw_matches_["HomeTeamID"] = sw_matches_["HomeTeam"].map(sw_to_id) 
            sw_matches_["AwayTeamID"] = sw_matches_["AwayTeam"].map(sw_to_id) 
            sw_matches_ = sw_matches_.rename(columns=lambda col: col if col in ["IdSW", "HomeTeamID", "AwayTeamID"] else f"{col}SW")

            # SS
            ss_matches_ = ss_matches[(ss_matches["League"] == league_slug) & (ss_matches["Season"] == str(season))].copy()
            ss_matches_["HomeTeamID"] = ss_matches_["HomeTeam"].map(ss_to_id) 
            ss_matches_["AwayTeamID"] = ss_matches_["AwayTeam"].map(ss_to_id)
            ss_matches_ = ss_matches_.rename(columns=lambda col: col if col in ["IdSS", "HomeTeamID", "AwayTeamID"] else f"{col}SS") 

            # Merge de los equipos
            matches_df = ss_matches_.merge(sw_matches_, on=["HomeTeamID", "AwayTeamID"], how="outer").drop_duplicates(subset="IdSS")
            matches_df["League"] = league_slug
            matches_df["Season"] = season
            all_matches_list.append(matches_df)

    # Dataframe con todos los partidos
    all_matches_df = pd.concat(all_matches_list)

    return clean_matched_matches_df(all_matches_df)

In [3]:
# Procesado de estadios
ss_venues = ss.ss_venues_clean()

# Procesado de entrenadores
ss_managers = ss.ss_managers_clean()
sw_managers = sw.sw_managers_clean()

# Procesado de equipos
ss_teams = ss.ss_teams_clean().drop_duplicates(subset="IdSS")
sw_teams = sw.sw_teams_clean().drop_duplicates(subset="IdSW")

# Información sobre los partidos
ss_matches = ss.ss_matches_clean()
sw_matches = sw.sw_matches_clean()

# Información sobre los jugadores
ss_players = ss.ss_players_clean()
sw_players = sw.sw_players_clean()

# Limpieza de venues dataframe
venues_df = clean_venues_df(df = ss_venues)

# Equipos matched a partir de partidos y equipos
teams_df = matched_teams_df(ss_matches=ss_matches, sw_matches=sw_matches, ss_teams=ss_teams, sw_teams=sw_teams)

# Procesado de jugadores a partir de jugadores y equipos (ya mapeados)
players_df = matched_players_df(ss_players=ss_players, sw_players=sw_players, teams_df=teams_df)

# Obtención del dataframe de entrenadores
managers_df = matched_managers_df(ss_managers=ss_managers, sw_managers=sw_managers)

# Obtención del dataframe de partidos
matches_df = matched_matches_df(ss_matches=ss_matches, sw_matches=sw_matches, teams_df=teams_df)

In [ ]:
# Path con información de los partidos
matches_path = os.path.join(RAW_DATA_PATH, "matches")

# Listas a concatenar
stats_list = []
lineups_list = []

# Diccionario con IDs de codigos de jugadores
ss_dict = players_df.dropna(subset="IdSS").set_index("IdSS")["ID"].to_dict()
sw_dict = players_df.dropna(subset="IdSW").set_index("IdSW")["ID"].to_dict()

matches_df = matches_df[matches_df["ID"] == "O0Z53"]

# Para cada partido, miramos de obtener información
for _, row in matches_df.iterrows():

    match_id = row["ID"]
    home_team = row["HomeTeam"]
    away_team = row["AwayTeam"]

    # Información que necesitaremos
    league_slug = row["League"]
    season = row["Season"]
    id_ss = row["IdSS"]
    id_sw = row["IdSW"]

    # Definición de paths
    path_lineups_ss = os.path.join(matches_path, f"{league_slug}_{season}", f"{int(id_ss)}_lineups.json") if pd.notna(id_ss) else None
    path_stats_ss = os.path.join(matches_path, f"{league_slug}_{season}", f"{int(id_ss)}_stats.json") if pd.notna(id_ss) else None
    path_sw = os.path.join(matches_path, f"{league_slug}_{season}", f"{id_sw}.json") if pd.notna(id_sw) else None

    # Lineups SW si no es none
    if path_sw is not None and os.path.exists(path_sw):
        sw_lineups_df, home_avg_age, away_avg_age = sw.sw_lineups_single_match(path_sw=path_sw, match_id=match_id, home_team_id=home_team, away_team_id=away_team)
        sw_lineups_df["player"] = sw_lineups_df["player"].map(sw_dict)
        sw_lineups_df = sw_lineups_df.rename(columns=lambda col: col if col in ["Match", "team", "player"] else f"{col}SW")
    else:
        sw_lineups_df = None
        home_avg_age = None
        away_avg_age = None

    # Lineups SS si no es none
    if path_lineups_ss is not None and os.path.exists(path_lineups_ss):
        ss_lineups_df, home_formation, away_formation = ss.ss_lineups_single_match(path_lineups_ss=path_lineups_ss, match_id=match_id, home_team_id=home_team, away_team_id=away_team)
        ss_lineups_df["player"] = ss_lineups_df["player"].map(ss_dict)
        ss_lineups_df = ss_lineups_df.rename(columns=lambda col: col if col in ["Match", "team", "player"] else f"{col}SS")
    else:
        ss_lineups_df = None
        home_formation = None
        away_formation = None

    # Stats SS si no es none
    if path_stats_ss is not None and os.path.exists(path_stats_ss):
        ss_stats_df = ss.ss_stats_single_match(path_stats_ss=path_stats_ss, match_id=match_id, home_team_id=home_team, away_team_id=away_team)
        ss_stats_df.insert(2, "Formation", [home_formation, away_formation])        # Añadimos formaciones
        ss_stats_df.insert(3, "AvgAge", [home_avg_age, away_avg_age])               # Añadimos edad media
        stats_list.append(ss_stats_df)

    # Concatenamos dataframe de alineaciones
    if ss_lineups_df is None and sw_lineups_df is None:
        lineups_df = None
    elif ss_lineups_df is None:
        lineups_df = sw_lineups_df
    elif sw_lineups_df is None:
        lineups_df = ss_lineups_df
    else:
        lineups_df = ss_lineups_df.merge(sw_lineups_df, on=["Match", "team", "player"], how="outer").drop_duplicates(subset="player")

    # Limpieza del dataframe de alineaciones
    # CLEANING
    lineups_list.append(lineups_df)

# Concatenamos dataframes para obtener las estadísticas
all_lineups_df = pd.concat(lineups_list, ignore_index=True)    
all_stats_df = pd.concat(stats_list, ignore_index=True)             # Intentar afegir managers aquí per tenir stats

In [49]:
all_lineups_df

,Match,team,player,shirt_numSS,positionSS,totalPassSS,accuratePassSS,totalLongBallsSS,accurateLongBallsSS,goalAssistSS,...,redCardSW,secondYellowSW,blockedScoringAttSW,goalAssistSW,wonCornersSW,cornerTakenSW,ontargetScoringAttSW,yellowCardSW,goalsSW,totalOffsideSW
0,O0Z53,IG121,04DN0,7,M,16.0,11.0,1.0,0.0,0.0,...,0,0,2,0,0,2,0,0,0,0
1,O0Z53,IG121,2WWY7,23,M,34.0,31.0,2.0,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,O0Z53,IG121,5EPSM,13,G,35.0,14.0,25.0,4.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,O0Z53,IG121,6G0HA,2,D,20.0,18.0,1.0,1.0,0.0,...,0,0,1,0,0,0,0,0,0,0
4,O0Z53,IG121,6XPBV,12,F,17.0,14.0,3.0,1.0,0.0,...,0,0,0,0,2,1,1,0,0,0
5,O0Z53,IG121,8D5T5,9,F,15.0,10.0,1.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
6,O0Z53,IG121,J10OI,19,M,15.0,11.0,0.0,0.0,0.0,...,0,0,0,0,1,0,1,1,0,0
7,O0Z53,IG121,LRDV8,20,D,25.0,21.0,5.0,2.0,0.0,...,0,0,0,0,0,0,0,0,0,0
8,O0Z53,IG121,LYKYI,22,D,50.0,37.0,9.0,2.0,0.0,...,0,0,0,0,0,0,0,0,0,0
9,O0Z53,IG121,SM9FL,18,M,12.0,8.0,2.0,2.0,0.0,...,0,0,0,0,0,0,0,0,0,1


In [20]:
players_df[players_df["IdSW"] == "1k2svnzs4woe6gpegqlrxjjf8"]

,ID,Slug,Name,ShortName,LongName,FirstName,LastName,ShortFirstName,ShortLastName,MatchName,...,Height,PrefFoot,Position,FirstPos,SecondPos,ThirdPos,ContractUntil,MarketValue,IdSS,IdSW
7845,OA8UQ,elis_stefan_bishesari,Elis Ștefan Bishesari,Elis Bishesari,Elis Ștefan Bishesari,Elis Ștefan,Bishesari,Elis,Bishesari,E. Bishesari,...,<NA>,NaN,Goalkeeper,NaN,NaN,NaN,NaN,<NA>,NaN,1k2svnzs4woe6gpegqlrxjjf8


In [38]:
sw_lineups_df

In [43]:
lineups_df

,Match,team,player,shirt_numSS,positionSS,totalPassSS,accuratePassSS,totalLongBallsSS,accurateLongBallsSS,goalAssistSS,...,redCardSW,secondYellowSW,blockedScoringAttSW,goalAssistSW,wonCornersSW,cornerTakenSW,ontargetScoringAttSW,yellowCardSW,goalsSW,totalOffsideSW
0,O0Z53,IG121,04DN0,7,M,16.0,11.0,1.0,0.0,0.0,...,0,0,2,0,0,2,0,0,0,0
1,O0Z53,IG121,2WWY7,23,M,34.0,31.0,2.0,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,O0Z53,IG121,5EPSM,13,G,35.0,14.0,25.0,4.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,O0Z53,IG121,6G0HA,2,D,20.0,18.0,1.0,1.0,0.0,...,0,0,1,0,0,0,0,0,0,0
4,O0Z53,IG121,6XPBV,12,F,17.0,14.0,3.0,1.0,0.0,...,0,0,0,0,2,1,1,0,0,0
5,O0Z53,IG121,8D5T5,9,F,15.0,10.0,1.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
6,O0Z53,IG121,J10OI,19,M,15.0,11.0,0.0,0.0,0.0,...,0,0,0,0,1,0,1,1,0,0
7,O0Z53,IG121,LRDV8,20,D,25.0,21.0,5.0,2.0,0.0,...,0,0,0,0,0,0,0,0,0,0
8,O0Z53,IG121,LYKYI,22,D,50.0,37.0,9.0,2.0,0.0,...,0,0,0,0,0,0,0,0,0,0
9,O0Z53,IG121,SM9FL,18,M,12.0,8.0,2.0,2.0,0.0,...,0,0,0,0,0,0,0,0,0,1
